# example of read midas CUPID file
```
                +-------------------+
                |     Web (mhttpd)  |
                +---------+---------+
                          |
                          v
+---------+      +------------------+      +-----------+
|Frontend | ---> |   Event Buffer   | ---> |  Logger   |
+---------+      +------------------+      +-----------+
      |                    |
      |                    v
      |              +-----------+
      |              | Analyzer  |
      |              +-----------+
      |
      v
+-------------------+
|        ODB        |
+-------------------+
```
-  Frontend: Reads hardware → builds events (banks ).
-  Event Buffer (shared memory): Frontend sends events to buffer.
-  Consumers read buffer: Logger → writes data to disk, Analyzer → processes data online
-  ODB (Online Database): Stores configuration, run state, parameters (used by all components).
-  → Hardware → Frontend → Event Buffer → Logger / Analyzer → Disk
```
Event
 ├── Bank "DATA" → Raw Data
 ├── Bank "HEAD" → ...
 ├── Bank "TEFP" → ...
 ├── Bank "TEFP" → ...
 └── Bank "... " → ...
```

In [ ]:
import struct
import sys
from typing import Optional, Tuple

import midas.file_reader


def parse_packet(pkt: bytes) -> Optional[Tuple[int, int, int, int, bytes]]:
    """
    Ritorna:
        (ssrc, seq, ts, csrc0, payload_bytes)
    oppure None se il pacchetto non è compatibile.

    Assunzione: il contenuto della bank DATA è il pacchetto RTP completo:
        RTP header 12B + eventuale CSRC list + payload + trailing byte
    """
    if len(pkt) < 12:
        return None

    b0 = pkt[0]
    version = (b0 >> 6) & 0x03
    cc = b0 & 0x0F

    if version != 2:
        return None

    seq = struct.unpack_from(">H", pkt, 2)[0]
    ts = struct.unpack_from(">I", pkt, 4)[0]
    ssrc = struct.unpack_from(">I", pkt, 8)[0]

    header_len = 12 + 4 * cc
    if len(pkt) < header_len:
        return None

    csrc0 = 0
    if cc >= 1:
        csrc0 = struct.unpack_from(">I", pkt, 12)[0]

    payload = pkt[header_len:]
    return ssrc, seq, ts, csrc0, payload


def bank_to_bytes(bank_data) -> bytes:
    """
    Converte il contenuto della bank in bytes.
    Utile perché .data può essere bytes, bytearray, array, tuple, numpy array, ecc.
    """
    if bank_data is None:
        return b""

    if isinstance(bank_data, (bytes, bytearray)):
        return bytes(bank_data)

    try:
        return bytes(bank_data)
    except Exception:
        return bytes(int(x) & 0xFF for x in bank_data)


def decode_payload_samples(payload: bytes):
    """
    Decodifica payload con layout:
        [MSB][MID][LSB][channel_id]
    e un trailing byte finale da ignorare.
    """
    if len(payload) < 1:
        return []

    data = payload[:-1]   # scarta trailing byte
    nwords = len(data) // 4
    data = data[:nwords * 4]

    out = []
    for k in range(nwords):
        i = 4 * k
        msb = data[i + 0]
        mid = data[i + 1]
        lsb = data[i + 2]
        ch  = data[i + 3]

        u24 = (msb << 16) | (mid << 8) | lsb
        out.append((ch, u24))

    return out

In [ ]:
import midas.file_reader

run     = int(input("run number [int] "))
path    = '/home/mazzitel/nmv-data/CUPID/'
fname = ('/run%05d.mid.gz' % run)
mf = midas.file_reader.MidasFile(path+fname)

# returns an object containing the Begin-of-Run ODB snapshot, with the full ODB tree available as a nested dictionary in its .data attribute.  #
odb = mf.get_bor_odb_dump().data

try:
    Run_description   = odb['Experiment']['Run Parameters']['Run description']
    Run_number = odb["Runinfo"]["Run number"]
    print('Run_description: ', Run_description)
    print('run_number: ', Run_number)
except:
    print('WARNING: no run description')


In [ ]:
%matplotlib inline
import numpy as np
import midas.file_reader
from datetime import datetime
verbose = True

# load file
run     = int(input("run number [int] "))
path    = './'    
fname = ('/run%05d.mid.gz' % run)
mf = midas.file_reader.MidasFile(path+fname)

# get odb dump and retrive info  #######
odb = mf.get_bor_odb_dump().data

try:
    Run_description   = odb['Experiment']['Run Parameters']['Run description']
    Events_logged     = odb['Logger']['Channels']['0']['Statistics']['Events written']
    
    print('Run_description: ', Run_description)
    print('Events_logged: ', Events_logged)
except:
    print('WARNING: no run description')


for event in mf:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if verbose or event_number % 1000==0:
        print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("Event # %s at %s, banks %s" % (event_number, datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))

    for bank_name, bank in event.banks.items():
        if ('DATA' in bank_name): 
            raw = bank_to_bytes(event.banks["DATA"].data)

            parsed = parse_packet(raw)
            
            if parsed is None:
                print(f"Evento {iev}: DATA non compatibile con RTP")
                continue
    
            ssrc, seq, ts, csrc0, payload = parsed
    
            # print(
            #     f"Evento {event_number:6d}  "
            #     f"SSRC=0x{ssrc:08X}  "
            #     f"SEQ={seq:5d}  "
            #     f"TS={ts:10d}  "
            #     f"CSRC0=0x{csrc0:08X}  "
            #     f"payload_len={len(payload)}"
            # )        

            samples = decode_payload_samples(payload)
            
            print(
                f"Evento {event_number:6d}  "
                f"SSRC=0x{ssrc:08X}  "
                f"SEQ={seq:5d}  "
                f"TS={ts:10d}  "
                f"Nsamples={len(samples)}"
            )
            
            for ch, u24 in samples[:8]:
                print(f"    ch={ch:3d}  u24=0x{u24:06X}")

            
print('DONE')


In [ ]:
import struct
import time
from collections import deque
from dataclasses import dataclass
from typing import Deque, Dict, Optional, Tuple

import matplotlib.pyplot as plt
from IPython.display import clear_output, display

import midas.file_reader

MAX24 = 16777215.0  # 2^24 - 1


def u24_offset_to_volts(u24: int, full_scale_volts: float) -> float:
    norm = u24 / MAX24
    span = 2.0 * full_scale_volts
    return norm * span - full_scale_volts


@dataclass(frozen=True)
class StreamKey:
    ssrc: int
    channel_id: int


@dataclass
class StreamState:
    buf: Deque[float]
    last_seq: Optional[int] = None
    last_ts: Optional[int] = None
    last_seen: float = 0.0


def bank_to_bytes(bank_data) -> bytes:
    """Converte event.banks['DATA'].data in bytes."""
    if bank_data is None:
        return b""

    if isinstance(bank_data, (bytes, bytearray)):
        return bytes(bank_data)

    try:
        return bytes(bank_data)
    except Exception:
        return bytes(int(x) & 0xFF for x in bank_data)


def parse_packet(pkt: bytes) -> Optional[Tuple[int, int, int, int, bytes]]:
    """
    Ritorna:
        (ssrc, seq, ts, csrc0, payload)
    oppure None se non è un pacchetto RTP valido.
    """
    if len(pkt) < 12:
        return None

    b0 = pkt[0]
    version = (b0 >> 6) & 0x03
    cc = b0 & 0x0F

    if version != 2:
        return None

    seq = struct.unpack_from(">H", pkt, 2)[0]
    ts = struct.unpack_from(">I", pkt, 4)[0]
    ssrc = struct.unpack_from(">I", pkt, 8)[0]

    header_len = 12 + 4 * cc
    if len(pkt) < header_len:
        return None

    csrc0 = 0
    if cc >= 1:
        csrc0 = struct.unpack_from(">I", pkt, 12)[0]

    payload = pkt[header_len:]
    return ssrc, seq, ts, csrc0, payload


def ingest_payload(
    payload: bytes,
    ssrc: int,
    seq: Optional[int],
    ts: Optional[int],
    streams: Dict[StreamKey, StreamState],
    full_scale_volts: float,
    max_points: int,
):
    """
    Decodifica il payload con layout:
        [MSB][MID][LSB][channel_id]
    assumendo 1 trailing byte finale da scartare.
    """
    if len(payload) < 1:
        return

    data = payload[:-1]  # trailing byte
    nwords = len(data) // 4
    if nwords <= 0:
        return

    usable = data[: nwords * 4]
    mv = memoryview(usable)
    now = time.time()

    for k in range(nwords):
        base = k * 4
        msb = mv[base + 0]
        mid = mv[base + 1]
        lsb = mv[base + 2]
        ch_id = mv[base + 3]

        u24 = (msb << 16) | (mid << 8) | lsb
        v = u24_offset_to_volts(u24, full_scale_volts)

        key = StreamKey(ssrc=ssrc, channel_id=int(ch_id))
        st = streams.get(key)
        if st is None:
            st = StreamState(buf=deque(maxlen=max_points))
            streams[key] = st

        st.buf.append(v)
        st.last_seq = seq
        st.last_ts = ts
        st.last_seen = now

In [ ]:
def plot_streams_notebook(
    filename: str,
    fs_volts: float = 10.0,
    max_points: int = 5000,
    max_plots: int = 8,
    events_per_refresh: int = 200,
    refresh_every: float = 0.2,
    max_events: Optional[int] = None,
):
    """
    Legge un file MIDAS e aggiorna i plot in notebook.

    Parametri:
      filename           : file .mid o .mid.lz4
      fs_volts           : full-scale +/- volts
      max_points         : lunghezza buffer rolling per stream
      max_plots          : numero massimo di subplot
      events_per_refresh : quanti eventi leggere tra un refresh e il successivo
      refresh_every      : pausa tra refresh grafici
      max_events         : se non None, si ferma dopo questo numero di eventi letti
    """
    mf = midas.file_reader.MidasFile(filename)

    streams: Dict[StreamKey, StreamState] = {}
    plot_keys: list[StreamKey] = []

    fig = plt.figure(figsize=(12, 8))
    fig.suptitle("MIDAS logger RTP streams from DATA bank")

    ncols = 2 if max_plots > 1 else 1
    nrows = (max_plots + ncols - 1) // ncols

    axes = []
    lines = []
    titles = []

    for i in range(max_plots):
        ax = fig.add_subplot(nrows, ncols, i + 1)
        ax.set_xlabel("sample index (rolling)")
        ax.set_ylabel("V")
        ax.grid(True)
        (ln,) = ax.plot([], [])
        axes.append(ax)
        lines.append(ln)
        titles.append(ax.set_title("(empty)"))

    processed_total = 0
    skipped_no_data = 0
    skipped_bad_rtp = 0

    try:
        for ev in mf:
            if max_events is not None and processed_total >= max_events:
                break

            processed_total += 1

            if not hasattr(ev, "banks") or "DATA" not in ev.banks:
                skipped_no_data += 1
                continue

            raw = bank_to_bytes(ev.banks["DATA"].data)
            parsed = parse_packet(raw)
            if parsed is None:
                skipped_bad_rtp += 1
                continue

            ssrc, seq, ts, csrc0, payload = parsed

            ingest_payload(
                payload=payload,
                ssrc=ssrc,
                seq=seq,
                ts=ts,
                streams=streams,
                full_scale_volts=fs_volts,
                max_points=max_points,
            )

            for key in streams.keys():
                if key not in plot_keys and len(plot_keys) < max_plots:
                    plot_keys.append(key)

            if processed_total % events_per_refresh == 0:
                for i in range(max_plots):
                    ax = axes[i]
                    ln = lines[i]

                    if i >= len(plot_keys):
                        ln.set_data([], [])
                        titles[i].set_text("(empty)")
                        ax.relim()
                        ax.autoscale_view()
                        continue

                    key = plot_keys[i]
                    st = streams.get(key)

                    if st is None:
                        ln.set_data([], [])
                        titles[i].set_text("(gone)")
                        ax.relim()
                        ax.autoscale_view()
                        continue

                    y = list(st.buf)
                    x = list(range(len(y)))

                    ln.set_data(x, y)
                    titles[i].set_text(
                        f"SSRC=0x{key.ssrc:08X} ch={key.channel_id}  "
                        f"n={len(y)} seq={st.last_seq} ts={st.last_ts}"
                    )
                    ax.relim()
                    ax.autoscale_view()

                clear_output(wait=True)
                display(fig)
                print(f"Eventi letti: {processed_total}")
                print(f"Eventi senza DATA: {skipped_no_data}")
                print(f"DATA non RTP: {skipped_bad_rtp}")
                time.sleep(refresh_every)

        # refresh finale
        for i in range(max_plots):
            ax = axes[i]
            ln = lines[i]

            if i >= len(plot_keys):
                ln.set_data([], [])
                titles[i].set_text("(empty)")
                ax.relim()
                ax.autoscale_view()
                continue

            key = plot_keys[i]
            st = streams.get(key)

            if st is None:
                ln.set_data([], [])
                titles[i].set_text("(gone)")
                ax.relim()
                ax.autoscale_view()
                continue

            y = list(st.buf)
            x = list(range(len(y)))

            ln.set_data(x, y)
            titles[i].set_text(
                f"SSRC=0x{key.ssrc:08X} ch={key.channel_id}  "
                f"n={len(y)} seq={st.last_seq} ts={st.last_ts}"
            )
            ax.relim()
            ax.autoscale_view()

        clear_output(wait=True)
        display(fig)
        print("Lettura completata.")
        print(f"Eventi letti: {processed_total}")
        print(f"Eventi senza DATA: {skipped_no_data}")
        print(f"DATA non RTP: {skipped_bad_rtp}")

    finally:
        try:
            mf.close()
        except Exception:
            pass

In [ ]:
run     = int(input("run number [int] "))
path    = './'    
fname = (path+'run%05d.mid.gz' % run)
plot_streams_notebook(
    filename=fname,
    fs_volts=10.0,
    max_points=5000,
    max_plots=8,
    events_per_refresh=200,
    refresh_every=0.1,
    max_events=None,
)

In [ ]:
%matplotlib inline
import numpy as np
import midas.file_reader
from datetime import datetime
verbose = True

# load file
run     = int(input("run number [int] "))
path    = '/home/mazzitel/nmv-data/CUPID'    
fname = ('/run%05d.mid.gz' % run)
mf = midas.file_reader.MidasFile(path+fname)

# get odb dump and retrive info  #######
odb = mf.get_bor_odb_dump().data

try:
    Run_description   = odb['Experiment']['Run Parameters']['Run description']
    Events_logged     = odb['Logger']['Channels']['0']['Statistics']['Events written']
    
    print('Run_description: ', Run_description)
    print('Events_logged: ', Events_logged)
except:
    print('WARNING: no run description')


for event in mf:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if verbose or event_number % 1000==0:
        print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("Event # %s at %s, banks %s" % (event_number, datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))

    for bank_name, bank in event.banks.items():
        if ('RTPH' in bank_name): 
            raw = bank_to_bytes(event.banks["RTPH"].data)

            parsed = parse_packet(raw)
            
            if parsed is None:
                print(f"Evento {event_number}: DATA non compatibile con RTP")
                continue
    
            ssrc, seq, ts, csrc0, payload = parsed
    
            # print(
            #     f"Evento {event_number:6d}  "
            #     f"SSRC=0x{ssrc:08X}  "
            #     f"SEQ={seq:5d}  "
            #     f"TS={ts:10d}  "
            #     f"CSRC0=0x{csrc0:08X}  "
            #     f"payload_len={len(payload)}"
            # )        

            samples = decode_payload_samples(payload)
            
            print(
                f"Evento {event_number:6d}  "
                f"SSRC=0x{ssrc:08X}  "
                f"SEQ={seq:5d}  "
                f"TS={ts:10d}  "
                f"Nsamples={len(samples)}"
            )
        if ('RTPP' in bank_name): 
            raw = bank_to_bytes(event.banks["RTPP"].data)
            for ch, u24 in samples[:8]:
                print(f"    ch={ch:3d}  u24=0x{u24:06X}")

            
print('DONE')
